##  Alien Dictionary

You are given a list of strings `words` from an alien language, sorted lexicographically according to the rules of this new language. The goal is to determine the order of the characters in the alien alphabet.

- Each word consists of lowercase English letters.
- The alien alphabet may not match the English alphabet.
- If the order is invalid or cannot be determined, return an empty string.

**Return:**
A string representing the characters in the correct order. If there are multiple valid orders, return any of them. If no valid order exists, return an empty string.

### 🧪 Example 1:
Input: words = ["wrt","wrf","er","ett","rftt"]

Output: "wertf"

### 🧪 Example 2:
Input: words = ["z","x"]

Output: "zx"

### 🧪 Example 3:
Input: words = ["z","x","z"]

Output: "" (No valid order)

**Constraints:**
- 1 <= words.length <= 100
- 1 <= words[i].length <= 100
- words[i] consists of only lowercase English letters

In [ ]:
from typing import List, Dict, Set
from collections import defaultdict

UNVISITED = 0
VISITING = 1
VISITED = 2

class Solution:
    def alienOrder(self, words: List[str]) -> str:
        # 1. Construir o grafo de precedência
        graph = self.build_graph(words)
        if graph is None:
            return ""  # Ordem inválida
        # 2. Ordenação topológica com DFS recursivo + visited
        return self.topological_sort_dfs(graph)

    def build_graph(self, words: List[str]) -> Dict[str, Set[str]]:
        """Extrai relações de precedência entre caracteres."""
        graph = defaultdict(set)
        all_chars = set(''.join(words))
        for char in all_chars:
            graph[char] = set()
        
        for i in range(len(words) - 1):
            w1, w2 = words[i], words[i+1]
            min_len = min(len(w1), len(w2))
            # Caso inválido: prefixo maior vem antes
            if w1[:min_len] == w2[:min_len] and len(w1) > len(w2):
                return None
            # Encontra a primeira diferença de caracteres
            # zip(w1, w2) agrupa caracteres em pares: [('w','w'), ('r','r'), ('t','f'), ...]
            for char_w1, char_w2 in zip(w1, w2):
               if char_w1 != char_w2:
                    # char_w1 vem antes de char_w2
                    graph[char_w1].add(char_w2)  # Aresta: a → b
                    break  # Para aqui, encontrou a primeira diferença 
        return graph

    def topological_sort_dfs(self, graph: Dict[str, Set[str]]) -> str:
        """
        DFS recursivo para ordenação topológica com detecção de ciclos.
        States: 0=unvisited, 1=visiting (em processamento), 2=visited (completo)
        """
        state = {char: UNVISITED for char in graph}
        result = []
        
        def dfs(node: str) -> bool:
            if state[node] == VISITING:
                return False  # CICLO: reentramos um nó em processamento
            if state[node] == VISITED:
                return True  # Já processado, skip
            
            state[node] = VISITING  # Marca como "visitando"
            # Explora todos os vizinhos primeiro (postorder)
            for neighbor in graph[node]:
                if not dfs(neighbor):
                    return False
            
            state[node] = VISITED  # Marca como "visitado"
            result.append(node)  # Adiciona APÓS explorar vizinhos (postorder)
            return True  # O(V + E)
        
        # Inicia DFS para cada nó não visitado
        for char in graph:
            if state[char] == UNVISITED:
                if not dfs(char):
                    return ""
        
        # Reverso para obter ordem topológica
        return ''.join(result[::-1])  # O(V)

wertf
zx



In [ ]:

from typing import List, Set, Dict, Optional
from collections import deque
class Solution:
    def alienOrder(self,words:List[str]) -> str:
        adjacency_list: Dict[str, Set[str]] = {}
        in_degree: Dict[str,int] = {}
        def build_graph() -> bool:
            for i in range(0, len(words) - 1):
                word1, word2 = words[i], words[i+1]
                if len(word1) > len(word2) and word1.startswith(word2):
                    return False
                for char1, char2 in zip(word1, word2):
                    if char1 != char2:
                        if char2 not in adjacency_list[char2]:
                            adjacency_list[char1].add(char2)
                            in_degree[char2] += 1
                        break 
            return True
        def topological_sort()-> str:
            result = []
            queue = deque([char for char in in_degree if in_degree[char] == 0])
            while queue:
                current = queue.popleft()
                result.append(current)
            
                neighbors = adjacency_list[current]
                for neighbor in neighbors:
                    in_degree[neighbor] -= 1
                    if in_degree[neighbor] == 0:
                        queue.append(neighbor)
            if len(result) < len(in_degree):
                return ''
            return ''.join(result)
        graph = build_graph()
        if not graph:
            return ''
        return topological_sort()

In [ ]:

# 🧪 Teste da Solução

sol = Solution()

# Exemplo 1
test1 = ["wrt","wrf","er","ett","rftt"]
result1 = sol.alienOrder(test1)
print(f"Test 1: {test1}")
print(f"Expected: 'wertf'")
print(f"Got:      '{result1}'")
print(f"✓ PASS" if result1 == "wertf" else f"✗ FAIL")
print()

# Exemplo 2
test2 = ["z","x"]
result2 = sol.alienOrder(test2)
print(f"Test 2: {test2}")
print(f"Expected: 'zx'")
print(f"Got:      '{result2}'")
print(f"✓ PASS" if result2 == "zx" else f"✗ FAIL")
print()

# Exemplo 3 (Ciclo / Inválido)
test3 = ["z","x","z"]
result3 = sol.alienOrder(test3)
print(f"Test 3: {test3}")
print(f"Expected: ''")
print(f"Got:      '{result3}'")
print(f"✓ PASS" if result3 == "" else f"✗ FAIL")


# Specification (Especificação de Engenharia)


O problema consiste em reconstruir a ordem alfabética de um idioma alienígena a partir de uma lista de palavras ordenadas lexicograficamente, modelando as precedências entre caracterrafes como um Go Direcionado.

 O objetivo final é realizar uma Ordenação Topológica sobre esse grafo ou identificar se a sequência fornecida é logicamente impossível (presença de ciclos ou violação de prefixos).


## Edge cases

- Para word1 e word2 ex: spy, spi conseguimos saber que y -> i
- Para word1 e word2 ex: spie, spy podemos retornar False para a ordem topologica, pois não pode ter uma palavra maior antes de palavra menor -> return ''
- O grafo pode ter mais de um bloco somente de ordem topocial 


# Planing

- Criar lista de adjacency do grafo ordenado
- retornar a lista topologica em forma de str dos caracters

words = ["wrt", "wrf", "er", "ett", "rft"]

wrt vs wrf 

varremos as words pegamos a word1 e a posterior word2 (0, len(word)-1) e usamos um for char1,char2 in zip()
Enquanto as letras sao iguais vamos comparando o char1  char2  caso for diferente temos t -> s


Podemos criar uma lista de adjacencia de usando a chave do dic como str o char e Set oque vem depois dele  [char, set(succesores)]
Podemos usar o in dregree para saber a ordem de pre requisitos e se algum char ter in dregree 0 quer dizer que não há depedencia anterior a ele (kahn)
Podemos fazer a ordem topologica, e retonar a ordem lexicofica final dos caracters

# blueprint

```py
   from typing import List, Set, Dict, Optional

    class Solution:
        def alienOrder(self,words:List[str]) -> str:
            adjacency_list: Dict[str, Set[str]] = {}
            in_degree: Dict[str,int] = {}
            def build_graph() -> bool:
                for i in range(en(words) - 1):
                    word1, word2 = words[i], words[i+1]
                    if len(word1) > len(word2) and w1.startswith(w2):
                        return False
                    for char1, char2 in zip(word1, word2):
                        if char1 != char2:
                            if char2 not in adjacency_list[char2]:
                                adjacency_list[char1].add(char2)
                                in_degree[char2] += 1
                            break 
                return True

            def topological_sort()-> str:
                result = []
                queue = deque([char for char in in_degree if in_degree[char] == 0])

                while queue:
                    current = queue.popleft()
                    result.append(current)
                
                    neighbors = adjacency_list[current]
                    for neighbor in neighbors:
                        in_degree[neighbor] -= 1
                        if in_degree[neighbor] == 0:
                            queue.append(neighbor)
                if len(result) < len(in_degree):
                    return ''
                return ''.join(result)

            graph = build_graph()
            if not graph:
                return ""
            return topological_sort()
```




##  Alien Dictionary

You are given a list of strings `words` from an alien language, sorted lexicographically according to the rules of this new language. The goal is to determine the order of the characters in the alien alphabet.

- Each word consists of lowercase English letters.
- The alien alphabet may not match the English alphabet.
- If the order is invalid or cannot be determined, return an empty string.

**Return:**
A string representing the characters in the correct order. If there are multiple valid orders, return any of them. If no valid order exists, return an empty string.

### 🧪 Example 1:
Input: words = ["wrt","wrf","er","ett","rftt"]

Output: "wertf"

### 🧪 Example 2:
Input: words = ["z","x"]

Output: "zx"

### 🧪 Example 3:
Input: words = ["z","x","z"]

Output: "" (No valid order)

**Constraints:**
- 1 <= words.length <= 100
- 1 <= words[i].length <= 100
- words[i] consists of only lowercase English letters

You are given a list of strings `words` 
from an alien language, sorted lexicographically according to the rules of this new language. 
The goal is to determine the order of the characters in the alien alphabet.

- Each word consists of lowercase English letters.
- The alien alphabet may not match the English alphabet.
- If the order is invalid or cannot be determined, return an empty string.

["wrt","wrf","er","ett","rftt"]

t -> f
f -> e
w -> e
r -> t
e -> r

# Plan 

Existe duas formas de resolver esse exercicio:

1) usando dfs postorder para dectação de ciclo e states constants, e append result o valor invertido sera a ordem lexigrafica ( L -> R -> Node)
2) Usando BFS com queue com algoritimo de Kahn 
        armazenando em um dict inOrder, inorder responde, Quantos elementos temos antes de mim ?
        guardiamos em outro dict os key prerequisite e values lista de cursos como representação do grafo topoligo.

3) Usando DFS, usariamos postorder com empilhamento recursivo, gerenciariamos os estados, como constantes, assim poderiamos descobrir se ha ciclo
    - Obs o result que sempre é appended antes do return final da recursão deve ter invertido [::-1], pois DFS fazemos folhas -> raiz bottom up
    - Fariamos um dict de course -> pre para representar o grafo topologico



In [ ]:
from typing import List, Dict
from collections import deque, defaultdict

class Solution:
    def alienOrder(self, words: List[str]) -> str:
        # 1. Preparar estruturas com TODOS os caracteres
        adj = defaultdict(list)
        # Inicializa in-degree 0 para cada letra única encontrada
        in_degree = {char: 0 for word in words for char in word}
        
        # 2. Construir o Grafo
        def build_graph() -> bool:
            for i in range(1, len(words)):
                word1, word2 = words[i-1], words[i]
                
                # Caso especial de prefixo inválido
                if len(word1) > len(word2) and word1.startswith(word2):
                    return False
                
                for c1, c2 in zip(word1, word2):
                    if c1 != c2:
                        # Só adiciona se a aresta for inédita para não inflar in-degree
                        if c2 not in adj[c1]:
                            adj[c1].append(c2)
                            in_degree[c2] += 1
                        break # Encontrou a diferença, para a comparação do par
            return True

        if not build_graph():
            return ""

        # 3. Topological Sort (Kahn's)
        queue = deque([char for char in in_degree if in_degree[char] == 0])
        result = []
        
        while queue:
            node = queue.popleft()
            result.append(node)
            for neighbor in adj[node]:
                in_degree[neighbor] -= 1
                if in_degree[neighbor] == 0:
                    queue.append(neighbor)
        
        # Se o resultado tem todas as letras, não houve ciclo
        return "".join(result) if len(result) == len(in_degree) else ""